# Avance 3. Baseline

## Equipo 41

**Proyecto:** Detección de ocultamiento de mercancía en video para retail

### Objetivo

Este notebook documenta el desarrollo y evaluación del modelo baseline utilizado como referencia para los experimentos posteriores del proyecto integrador. El objetivo es establecer un punto de comparación reproducible que permita evaluar la viabilidad de distinguir eventos de `ocultamiento` y `sin_ocultamiento` a partir de clips de video.

Debido a que el problema corresponde al reconocimiento de acciones en video, se utiliza una arquitectura R3D-18 implementada en PyTorch y preentrenada sobre Kinetics-400. Esta arquitectura permite procesar directamente secuencias RGB y aprender representaciones espacio-temporales a partir de la información visual y del movimiento presente entre frames. El modelo se adapta al problema mediante transferencia de aprendizaje y ajuste fino para clasificación binaria.

El desarrollo del baseline sigue la fase de modelado de CRISP-ML(Q), considerando tanto el desempeño predictivo como aspectos relacionados con generalización, complejidad, explicabilidad y demanda de recursos. La evaluación se realiza sobre conjuntos separados de entrenamiento, validación y prueba, manteniendo controles para evitar fuga de información entre clips relacionados y conservando la fuente como variable de control para analizar generalización.

Además del desempeño del modelo, este notebook analiza las representaciones aprendidas por R3D-18, el posible subajuste o sobreajuste durante el entrenamiento, las métricas apropiadas para el problema de clasificación binaria y el nivel de desempeño alcanzado respecto a referencias básicas.

### Relación entre la rúbrica y el tratamiento del baseline de video

La rúbrica del Avance 3 está formulada de manera general para problemas de aprendizaje automático. En este proyecto, sus criterios se adaptan a un problema de clasificación binaria de acciones sobre datos de video no estructurados. Esta adaptación conserva el propósito de cada criterio, pero considera que las características utilizadas para la clasificación no corresponden a variables tabulares definidas manualmente, sino a representaciones espacio-temporales aprendidas por una red neuronal convolucional 3D.

| Criterio de la rúbrica | Aplicación en este notebook |
|---|---|
| Algoritmo | Selección y justificación de R3D-18 como modelo baseline para reconocimiento de acciones en video. Se consideran la naturaleza espacio-temporal de los datos, el tamaño disponible del conjunto de entrenamiento, el uso de pesos preentrenados en Kinetics-400, la complejidad del modelo, sus requerimientos computacionales y sus limitaciones de interpretabilidad. |
| Características importantes | Las características no se seleccionan directamente a partir de columnas tabulares. R3D-18 aprende representaciones espacio-temporales durante el procesamiento de los clips. Se extraen los embeddings generados antes de la capa de clasificación y se analiza su estructura mediante reducción de dimensionalidad y representación visual. |
| Subajuste / sobreajuste | Se compara la evolución de la función de pérdida y de las métricas de desempeño entre entrenamiento y validación. También se comparan los resultados obtenidos en entrenamiento, validación y prueba para identificar posibles brechas de generalización. |
| Métrica | La evaluación se adapta a un problema de clasificación binaria con desbalance de clases. Las métricas principales serán ROC-AUC, PR-AUC y EER, dando especial atención a PR-AUC por el comportamiento sobre la clase minoritaria. Se complementan con matriz de confusión, precision, recall, F1-score, specificity y accuracy para analizar el comportamiento a un umbral de decisión determinado. |
| Desempeño | El resultado del modelo se compara contra referencias básicas y contra el comportamiento esperado de un clasificador no informativo. El desempeño obtenido se utiliza como punto de referencia para determinar posteriormente si modelos alternativos proporcionan mejoras relevantes. |

La selección y evaluación del modelo se realiza siguiendo CRISP-ML(Q), por lo que el análisis no se limita a una única métrica predictiva. También se documentan aspectos relacionados con generalización, tiempo de entrenamiento, recursos utilizados, complejidad y capacidad de interpretación de las representaciones aprendidas.

## 1. Definición del problema de modelado

El proyecto se formula como un problema de aprendizaje supervisado para clasificación binaria de acciones en video. El objetivo del modelo es distinguir entre clips que contienen un evento observable de `ocultamiento` de mercancía y clips que representan comportamiento `sin_ocultamiento`.

La unidad de análisis es un clip temporal de video y no el video completo. Esta decisión es necesaria porque un video puede contener diferentes acciones a lo largo del tiempo y un evento de ocultamiento puede representar solamente una fracción de su duración. En consecuencia, la etiqueta asignada a un video no se transfiere automáticamente a todos sus fragmentos.

Para el baseline, cada observación se representa mediante una secuencia RGB de 16 frames con resolución espacial de 112 × 112 píxeles. La entrada esperada por el modelo tiene la forma:

$$
X_i \in \mathbb{R}^{3 \times 16 \times 112 \times 112}
$$

La variable objetivo se define como:

$$
y_i =
\begin{cases}
0, & \text{sin\_ocultamiento} \\
1, & \text{ocultamiento}
\end{cases}
$$

La clase `ocultamiento` corresponde a clips en los que existe evidencia visual suficiente de acercamiento o introducción de mercancía hacia una zona definida de ocultamiento, como bolsillo, ropa, bolsa o mochila. La clase `sin_ocultamiento` incluye comportamientos observables donde la mercancía permanece visible o no se presenta la acción objetivo.

Los clips cuya acción no pueda determinarse con suficiente claridad debido a oclusiones, visibilidad insuficiente, cortes temporales u otras condiciones ambiguas no se fuerzan a una de las dos clases para el entrenamiento principal. Estos casos se conservan, cuando resulte pertinente, para análisis posterior de errores y limitaciones.

El modelo genera dos valores de salida asociados con las clases del problema. Durante el entrenamiento estos valores se utilizan como logits para calcular la función de pérdida y, durante la evaluación, pueden transformarse mediante Softmax en probabilidades estimadas para cada clase.

### 1.1 Decisiones de modelado y criterios de calidad

La definición del baseline considera las características de los datos, las restricciones del proyecto y el propósito de utilizar el modelo como referencia para experimentos posteriores. De acuerdo con CRISP-ML(Q), la selección y evaluación del modelo debe considerar tanto el desempeño predictivo como su capacidad de generalización y las restricciones relevantes para su posible utilización.

| Elemento | Decisión para el baseline |
|---|---|
| Tipo de aprendizaje | Supervisado |
| Tipo de problema | Clasificación binaria de acciones en video |
| Clase positiva | `ocultamiento` |
| Clase negativa | `sin_ocultamiento` |
| Unidad de análisis | Clip temporal |
| Entrada | Video RGB |
| Frames por muestra | 16 |
| Resolución de entrada | 112 × 112 |
| Modelo baseline | R3D-18 |
| Framework | PyTorch / Torchvision |
| Inicialización | Pesos preentrenados en Kinetics-400 |
| Estrategia de entrenamiento | Transfer learning y ajuste fino |
| Salida durante entrenamiento | Dos logits, uno por clase |
| Salida para evaluación | Probabilidades obtenidas mediante Softmax |
| Separación de datos | Entrenamiento, validación y prueba sin compartir entre particiones clips pertenecientes al mismo grupo (dataset, video_id) |
| Tratamiento de casos ambiguos | Exclusión del entrenamiento principal y conservación para análisis cuando sea pertinente |
| Uso previsto | Referencia experimental para comparar modelos posteriores |
| Alcance | Evaluación sobre videos grabados; no corresponde a un sistema productivo |

Además del desempeño predictivo, se consideran los siguientes criterios de calidad:

- **Generalización:** diferencia entre el desempeño obtenido en entrenamiento, validación y prueba.
- **Explicabilidad:** análisis posterior de las representaciones espacio-temporales aprendidas y de los errores de clasificación.
- **Complejidad:** número de parámetros y características de la arquitectura.
- **Demanda de recursos:** tiempo de entrenamiento, memoria utilizada y velocidad de inferencia.
- **Reproducibilidad:** registro de configuración, semillas, particiones, hiperparámetros y versiones de las bibliotecas utilizadas.

La clasificación producida por el modelo se mantiene separada de la lógica operativa de generación de alertas. El baseline estima la probabilidad de cada clase para un clip; la selección posterior de un umbral de decisión permitirá analizar el compromiso entre detecciones correctas y falsas alarmas.

## 2. Protocolo experimental y partición de datos

El Avance 2 estableció las condiciones que deben conservarse durante el
modelado para evitar fuga de información y controlar los principales riesgos
identificados durante Data Preparation.

En particular:

- Únicamente participan los clips elegibles para clasificación binaria;
- Los clips pertenecientes al mismo grupo `(dataset, video_id)` deben
  permanecer dentro de una sola partición;
- La composición de las particiones debe conservar, en la medida de lo
  posible, representación de ambas clases y de las diferentes fuentes;
- Cualquier tratamiento del desbalance se realizará después de la
  partición y exclusivamente sobre entrenamiento;
- La asignación a entrenamiento, validación y prueba se persistirá para
  utilizar exactamente las mismas observaciones al comparar arquitecturas.

A partir de estas condiciones, en este avance se construye y valida la
partición experimental que utilizará el baseline R3D-18.

In [1]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


# ---------------------------------------------------------
# Detectar automáticamente la raíz del repositorio
# ---------------------------------------------------------
def find_repo_root():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/mnt/c/dev/MNA---Proyecto-Integrador"),
        Path("/opt/workspace/MNA---Proyecto-Integrador"),
    ]

    for candidate in candidates:
        data_dir = candidate / "Datos" / "Preparacion_Datasets"
        if data_dir.is_dir():
            return candidate

    raise FileNotFoundError(
        "No se pudo localizar la raíz del repositorio "
        "MNA---Proyecto-Integrador."
    )


REPO_DIR = find_repo_root()

# Directorio de trabajo para la preparación de datos
BASE_DIR = REPO_DIR / "Datos" / "Preparacion_Datasets"

# Archivos generados durante la preparación
MANIFEST_PATH = BASE_DIR / "dataset_manifest_todos.csv"
FEATURES_PATH = BASE_DIR / "dataset_consolidado_features.csv"
DURACIONES_PATH = BASE_DIR / "duraciones_dataset_consolidado.csv"


# ---------------------------------------------------------
# Dataset físico consolidado
# ---------------------------------------------------------
# Local:
#   BNEXT_CLIPS_DIR=/home/ajimenez/mnt/bnext_clips
#
# BNext:
#   fallback a /opt/workspace/datasets/procesados/clips_4s
CLIPS_DIR = Path(
    os.environ.get(
        "BNEXT_CLIPS_DIR",
        "/opt/workspace/datasets/procesados/clips_4s"
    )
)


print("Repositorio:             ", REPO_DIR)
print("BASE_DIR:                ", BASE_DIR)
print("Manifest disponible:     ", MANIFEST_PATH.exists())
print("Features disponibles:    ", FEATURES_PATH.exists())
print("Duraciones disponibles:  ", DURACIONES_PATH.exists())
print("Directorio de clips:     ", CLIPS_DIR.exists())


# El archivo de features será la tabla principal para definir
# el protocolo experimental del baseline
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado: {FEATURES_PATH}"
    )

df = pd.read_csv(FEATURES_PATH)

print(f"\nRegistros: {len(df):,}")
print(f"Columnas:  {len(df.columns)}")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nPrimeros registros:")
display(df.head())

Repositorio:              /mnt/c/dev/MNA---Proyecto-Integrador
BASE_DIR:                 /mnt/c/dev/MNA---Proyecto-Integrador/Datos/Preparacion_Datasets
Manifest disponible:      True
Features disponibles:     True
Duraciones disponibles:   True
Directorio de clips:      True

Registros: 987
Columnas:  19

Columnas disponibles:
['dataset', 'video_id', 'video_origen', 'clip_id', 'inicio_s', 'fin_s', 'etiqueta', 'persona_id', 'evento_id', 'nota', 'nombre_clip', 'duracion_real_s', 'duracion_objetivo_s', 'diferencia_vs_4s', 'cumple_4s', 'requiere_normalizacion_temporal', 'elegible_modelo_binario', 'motivo_no_elegible', 'clase_binaria']

Primeros registros:


,dataset,video_id,video_origen,clip_id,inicio_s,fin_s,etiqueta,persona_id,evento_id,nota,nombre_clip,duracion_real_s,duracion_objetivo_s,diferencia_vs_4s,cumple_4s,requiere_normalizacion_temporal,elegible_modelo_binario,motivo_no_elegible,clase_binaria
0,CCTV_YOLO,CCTV_001,not_shoplifting1.mp4,CCTV_001,2.0,6.0,sin_ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_001__CCTV_001.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
1,CCTV_YOLO,CCTV_002,not_shoplifting2.mp4,CCTV_002,2.0,6.0,sin_ocultamiento,P002_AJ,NaN,NaN,CCTV_YOLO__CCTV_002__CCTV_002.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
2,CCTV_YOLO,CCTV_003,not_shoplifting3.mp4,CCTV_003,2.0,6.0,sin_ocultamiento,P003_AJ,NaN,NaN,CCTV_YOLO__CCTV_003__CCTV_003.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
3,CCTV_YOLO,CCTV_004,not_shoplifting4.mp4,CCTV_004,2.0,6.0,sin_ocultamiento,P004_AJ,NaN,NaN,CCTV_YOLO__CCTV_004__CCTV_004.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
4,CCTV_YOLO,CCTV_005,shoplifting1.mp4,CCTV_005,2.0,6.0,ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_005__CCTV_005.mp4,4.0,4.0,0.0,True,False,True,NaN,1.0


### 2.1 Unidad de agrupación

El Avance 2 definió la combinación `(dataset, video_id)` como unidad mínima
de agrupación para prevenir fuga de información entre clips relacionados.
Esta regla se conserva durante la construcción de las particiones.

Antes de generar el split definitivo se verifica adicionalmente `persona_id`,
ya que el Avance 2 estableció que los identificadores confiables de persona,
evento o escenario deben considerarse cuando permitan detectar dependencia
entre observaciones pertenecientes a videos distintos.

Para el modelado se utilizan únicamente registros con
`elegible_modelo_binario = True` y una clase binaria definida.

In [2]:
# Subconjunto utilizado para el baseline
df_modelo = df[
    df["elegible_modelo_binario"].eq(True)
    & df["clase_binaria"].notna()
].copy()

df_modelo["clase_binaria"] = (
    df_modelo["clase_binaria"].astype(int)
)

df_modelo["grupo_video"] = (
    df_modelo["dataset"].astype(str)
    + "__"
    + df_modelo["video_id"].astype(str)
)

print(f"Clips elegibles: {len(df_modelo):,}")
print(f"Grupos de video: {df_modelo['grupo_video'].nunique():,}")

assert len(df_modelo) == 985
assert set(df_modelo["clase_binaria"].unique()) == {0, 1}

Clips elegibles: 985
Grupos de video: 229


In [3]:
# Verificar cuántas clases aparecen dentro de cada video
clases_por_video = (
    df_modelo
    .groupby("grupo_video")["clase_binaria"]
    .nunique()
)

videos_multiclase = clases_por_video[
    clases_por_video > 1
]

print(
    "Videos que contienen clips de más de una clase:",
    len(videos_multiclase)
)

Videos que contienen clips de más de una clase: 16


### 2.2 Auditoría de identificadores de persona

El Avance 2 estableció que `(dataset, video_id)` constituye la unidad mínima
de agrupación para evitar fuga de información entre clips relacionados.
También se determinó que identificadores confiables de persona, evento o
escenario deben considerarse cuando permitan detectar dependencia entre
videos diferentes.

Antes de construir las particiones se revisa `persona_id` para determinar:

1. Qué proporción de los clips dispone de un identificador de persona;
2. Si los identificadores son consistentes dentro de cada fuente;
3. Si una misma persona aparece asociada con más de un grupo de video;
4. Si esas relaciones requieren ampliar la unidad de agrupación utilizada
   para separar entrenamiento, validación y prueba.

Los identificadores se interpretan únicamente dentro de cada dataset. Dos
valores iguales en fuentes diferentes no se consideran automáticamente la
misma persona, ya que cada fuente utiliza su propio esquema de identificación.
Los valores que representan identidad desconocida, como `PNA`, se excluyen
de este análisis.

In [4]:
# ------------------------------------------------------------
# Auditoría de persona_id antes de construir las particiones
# ------------------------------------------------------------

PERSONAS_DESCONOCIDAS = {
    "",
    "PNA",
    "P0NA",
    "NA",
    "N/A",
    "NONE",
    "UNKNOWN",
    "DESCONOCIDO"
}

df_personas = df_modelo.copy()

# Normalizar identificadores
df_personas["persona_id_limpio"] = (
    df_personas["persona_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Identificar registros con persona conocida
df_personas["persona_conocida"] = (
    ~df_personas["persona_id_limpio"]
    .str.upper()
    .isin(PERSONAS_DESCONOCIDAS)
)

print("Disponibilidad de persona_id por dataset:")
display(
    pd.crosstab(
        df_personas["dataset"],
        df_personas["persona_conocida"],
        margins=True
    ).rename(
        columns={
            False: "desconocida",
            True: "conocida"
        }
    )
)

# Trabajar únicamente con IDs conocidos
df_personas_conocidas = (
    df_personas[
        df_personas["persona_conocida"]
    ]
    .copy()
)

# El identificador se interpreta dentro de cada fuente
df_personas_conocidas["grupo_persona"] = (
    df_personas_conocidas["dataset"].astype(str)
    + "__"
    + df_personas_conocidas["persona_id_limpio"]
)

# Número de grupos de video asociados con cada persona
resumen_personas = (
    df_personas_conocidas
    .groupby(
        [
            "dataset",
            "grupo_persona",
            "persona_id_limpio"
        ],
        as_index=False
    )
    .agg(
        n_grupos_video=("grupo_video", "nunique"),
        n_clips=("clip_id", "size"),
        n_clases=("clase_binaria", "nunique")
    )
)

personas_multivideo = (
    resumen_personas[
        resumen_personas["n_grupos_video"] > 1
    ]
    .sort_values(
        ["n_grupos_video", "n_clips"],
        ascending=False
    )
)

print(
    "\nPersonas conocidas:",
    resumen_personas["grupo_persona"].nunique()
)

print(
    "Personas presentes en más de un grupo de video:",
    len(personas_multivideo)
)

if len(personas_multivideo) > 0:
    display(personas_multivideo)

    print("\nVideos asociados con personas multivideo:")

    detalle_multivideo = (
        df_personas_conocidas[
            df_personas_conocidas["grupo_persona"].isin(
                personas_multivideo["grupo_persona"]
            )
        ][
            [
                "dataset",
                "persona_id_limpio",
                "grupo_video",
                "video_origen",
                "clase_binaria"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "dataset",
                "persona_id_limpio",
                "grupo_video",
                "clase_binaria"
            ]
        )
    )

    display(detalle_multivideo)

Disponibilidad de persona_id por dataset:


persona_conocida,desconocida,conocida,All
dataset,,,
CCTV_YOLO,0,8,8
DCSASS,419,364,783
MNNIT,6,176,182
SINTETICO,0,12,12
All,425,560,985



Personas conocidas: 51
Personas presentes en más de un grupo de video: 16


,dataset,grupo_persona,persona_id_limpio,n_grupos_video,n_clips,n_clases
25,MNNIT,MNNIT__P001,P001,36,36,2
31,MNNIT,MNNIT__P007,P007,31,31,2
30,MNNIT,MNNIT__P006,P006,28,28,2
32,MNNIT,MNNIT__P008,P008,17,17,2
34,MNNIT,MNNIT__P010,P010,13,13,2
26,MNNIT,MNNIT__P002,P002,10,10,2
33,MNNIT,MNNIT__P009,P009,10,10,2
27,MNNIT,MNNIT__P003,P003,8,8,2
28,MNNIT,MNNIT__P004,P004,8,8,2
29,MNNIT,MNNIT__P005,P005,6,6,2



Videos asociados con personas multivideo:


,dataset,persona_id_limpio,grupo_video,video_origen,clase_binaria
0,CCTV_YOLO,P001_AJ,CCTV_YOLO__CCTV_001,not_shoplifting1.mp4,0
4,CCTV_YOLO,P001_AJ,CCTV_YOLO__CCTV_005,shoplifting1.mp4,1
1,CCTV_YOLO,P002_AJ,CCTV_YOLO__CCTV_002,not_shoplifting2.mp4,0
5,CCTV_YOLO,P002_AJ,CCTV_YOLO__CCTV_006,shoplifting2.mp4,1
2,CCTV_YOLO,P003_AJ,CCTV_YOLO__CCTV_003,not_shoplifting3.mp4,0
...,...,...,...,...,...
95,MNNIT,P012,MNNIT__shoplifting-89,shoplifting-89.mp4,1
96,MNNIT,P012,MNNIT__shoplifting-90,shoplifting-90.mp4,1
189,MNNIT,P014,MNNIT__normal-90,normal-90.mp4,0
98,MNNIT,P014,MNNIT__shoplifting-92,shoplifting-92.mp4,1


### 2.2.1 Resultado de la auditoría de persona

La auditoría confirmó que `persona_id` aporta información adicional sobre la
dependencia entre observaciones.

En particular, se identificaron personas conocidas que aparecen en varios
`video_id` dentro de una misma fuente. Este comportamiento es especialmente
relevante en MNNIT, donde algunos sujetos participan en múltiples videos de
ambas clases, y en CCTV_YOLO, donde una misma persona aparece en ejemplos de
`ocultamiento` y `sin_ocultamiento`.

Por esta razón, utilizar únicamente `(dataset, video_id)` como criterio de
separación podría colocar videos de una misma persona en entrenamiento y
evaluación. Esto mantendría separados los archivos de video, pero no impediría
que el modelo encontrara nuevamente al mismo sujeto.

La unidad definitiva de agrupación se construirá manteniendo juntos todos los
videos relacionados mediante identificadores de persona conocidos y
confiables. Los videos sin un identificador de persona disponible conservarán
como grupo mínimo la combinación `(dataset, video_id)`.

Los identificadores utilizados como marcadores de persona desconocida, como
`PNA` y `P0NA`, no se emplean para relacionar videos entre sí.

In [5]:
# ------------------------------------------------------------
# Construcción de grupos experimentales
# ------------------------------------------------------------

PERSONAS_DESCONOCIDAS = {
    "",
    "PNA",
    "P0NA",
    "NA",
    "N/A",
    "NONE",
    "UNKNOWN",
    "DESCONOCIDO"
}

df_split = df_modelo.copy()

df_split["persona_id_limpio"] = (
    df_split["persona_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_split["persona_conocida"] = (
    ~df_split["persona_id_limpio"]
    .str.upper()
    .isin(PERSONAS_DESCONOCIDAS)
)


# ------------------------------------------------------------
# Union-Find para conectar videos relacionados por persona
# ------------------------------------------------------------

parent = {}


def find(x):
    if x not in parent:
        parent[x] = x

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]


def union(a, b):
    raiz_a = find(a)
    raiz_b = find(b)

    if raiz_a != raiz_b:
        parent[raiz_b] = raiz_a


# Cada video comienza como un grupo independiente
for grupo_video in df_split["grupo_video"].unique():
    find(grupo_video)


# Dentro de cada dataset, unir videos que comparten
# una persona conocida
df_con_id = df_split[
    df_split["persona_conocida"]
].copy()

for (dataset, persona_id), grupo in df_con_id.groupby(
    ["dataset", "persona_id_limpio"]
):
    videos = grupo["grupo_video"].unique()

    if len(videos) > 1:
        primero = videos[0]

        for video in videos[1:]:
            union(primero, video)


# Aplicar compresión de caminos y generar IDs reproducibles
df_split["raiz_grupo"] = (
    df_split["grupo_video"]
    .map(find)
)

raices = sorted(
    df_split["raiz_grupo"].unique()
)

mapa_grupos = {
    raiz: f"G{idx:03d}"
    for idx, raiz in enumerate(raices, start=1)
}

df_split["grupo_experimental"] = (
    df_split["raiz_grupo"]
    .map(mapa_grupos)
)


# ------------------------------------------------------------
# Resumen de los grupos resultantes
# ------------------------------------------------------------

resumen_grupos = (
    df_split
    .groupby("grupo_experimental")
    .agg(
        n_clips=("clip_id", "size"),
        n_videos=("grupo_video", "nunique"),
        n_personas_conocidas=(
            "persona_id_limpio",
            lambda x: x[
                ~x.str.upper().isin(PERSONAS_DESCONOCIDAS)
            ].nunique()
        ),
        n_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        n_sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

print(f"Clips:                  {len(df_split):,}")
print(
    f"Videos originales:      "
    f"{df_split['grupo_video'].nunique():,}"
)
print(
    f"Grupos experimentales:  "
    f"{df_split['grupo_experimental'].nunique():,}"
)

print("\nTamaño de los grupos experimentales:")
display(
    resumen_grupos[
        ["n_clips", "n_videos"]
    ]
    .describe()
)

print("\nGrupos con más videos relacionados:")
display(
    resumen_grupos
    .sort_values(
        ["n_videos", "n_clips"],
        ascending=False
    )
    .head(15)
)

Clips:                  985
Videos originales:      229
Grupos experimentales:  63

Tamaño de los grupos experimentales:


,n_clips,n_videos
count,63.000000,63.000000
mean,15.634921,3.634921
std,13.993433,7.092253
min,1.000000,1.000000
25%,1.000000,1.000000
50%,13.000000,1.000000
75%,32.000000,1.500000
max,36.000000,36.000000



Grupos con más videos relacionados:


,grupo_experimental,n_clips,n_videos,n_personas_conocidas,n_ocultamiento,n_sin_ocultamiento
37,G038,36,36,1,17,19
38,G039,31,31,1,15,16
47,G048,28,28,1,12,16
39,G040,17,17,1,11,6
43,G044,13,13,1,4,9
40,G041,10,10,1,6,4
41,G042,10,10,1,5,5
42,G043,8,8,1,4,4
44,G045,8,8,1,6,2
46,G047,6,6,1,5,1


### 2.3 Construcción de las particiones experimentales

Antes de realizar la separación en entrenamiento, validación y prueba se
construyó una unidad denominada `grupo_experimental`. Su propósito es evitar
que observaciones estrechamente relacionadas aparezcan en particiones
diferentes y produzcan fuga de información.

Un grupo experimental reúne todos los clips que deben mantenerse juntos
durante la partición. Como condición mínima, todos los clips pertenecientes
al mismo `(dataset, video_id)` forman parte del mismo grupo. Adicionalmente,
cuando existe un `persona_id` conocido y confiable, se agrupan también los
videos de una misma fuente en los que aparece esa persona.

La relación se aplica de forma transitiva. Por ejemplo, si una persona aparece
en los videos A y B, y otra persona relaciona los videos B y C, los videos
A, B y C forman un único grupo experimental. De esta manera se evita que
videos relacionados indirectamente puedan quedar separados entre
entrenamiento y evaluación.

Los identificadores que representan una persona desconocida, como `PNA` o
`P0NA`, no se utilizan para unir videos. En esos casos se mantiene únicamente
la agrupación mínima definida por `(dataset, video_id)`.

Después de aplicar estas reglas, los 985 clips elegibles y 229 videos originales
quedaron organizados en 63 grupos experimentales indivisibles. Estos grupos,
y no los clips individuales, constituyen las unidades que se asignarán a las
particiones.

Esta estrategia es necesaria porque una partición aleatoria a nivel de clip,
o incluso únicamente a nivel de archivo de video, podría permitir que el
modelo observara durante el entrenamiento escenas o personas estrechamente
relacionadas con las utilizadas posteriormente para validación o prueba. En
ese caso, las métricas podrían reflejar parcialmente reconocimiento de la
persona, apariencia, escenario u otras características compartidas, en lugar
de generalización de la acción de ocultamiento.

Se adopta como objetivo una distribución aproximada de:

- 70 % para entrenamiento;
- 15 % para validación;
- 15 % para prueba.

Estas proporciones son objetivos y no restricciones exactas. La prioridad es
preservar la independencia entre los 63 grupos experimentales. Dentro de esta
restricción se busca que las particiones mantengan una composición
razonablemente similar respecto al número de clips, las dos clases y las
fuentes de datos.

La partición se obtiene mediante una búsqueda reproducible entre múltiples
asignaciones posibles de los grupos experimentales. Cada solución candidata
se evalúa de acuerdo con su desviación respecto a las proporciones objetivo
en número de clips, distribución de `ocultamiento` y `sin_ocultamiento`, y
representación de las fuentes.

El conjunto de prueba permanecerá aislado durante el desarrollo del modelo.
La selección de checkpoints, hiperparámetros y umbrales de decisión se
realizará utilizando únicamente entrenamiento y validación.

La asignación final se conservará como un artefacto reproducible para que
R3D-18 y las arquitecturas evaluadas posteriormente utilicen exactamente las
mismas particiones.

In [6]:
# ------------------------------------------------------------
# Resumen de información necesaria para el split
# ------------------------------------------------------------

resumen_split = (
    df_split
    .groupby("grupo_experimental")
    .agg(
        dataset=("dataset", "first"),
        n_clips=("clip_id", "size"),
        n_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        n_sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        ),
        n_videos=("grupo_video", "nunique")
    )
    .reset_index()
)

# Verificar que ningún grupo experimental mezcle fuentes
datasets_por_grupo = (
    df_split
    .groupby("grupo_experimental")["dataset"]
    .nunique()
)

assert datasets_por_grupo.max() == 1

print("Grupos experimentales por fuente:")
display(
    resumen_split["dataset"]
    .value_counts()
    .rename_axis("dataset")
    .to_frame("grupos")
)

print("\nClips contenidos en esos grupos:")
display(
    resumen_split
    .groupby("dataset")["n_clips"]
    .sum()
    .to_frame("clips")
)

Grupos experimentales por fuente:


,grupos
dataset,
DCSASS,27
MNNIT,20
SINTETICO,12
CCTV_YOLO,4



Clips contenidos en esos grupos:


,clips
dataset,
CCTV_YOLO,8
DCSASS,783
MNNIT,182
SINTETICO,12


In [7]:
import numpy as np

# ------------------------------------------------------------
# Configuración de la partición
# ------------------------------------------------------------

SPLITS = ("train", "val", "test")

PROPORCIONES = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15
}

SEED = 41
N_INTENTOS = 100000

rng = np.random.default_rng(SEED)


# ------------------------------------------------------------
# Totales globales y objetivos
# ------------------------------------------------------------

total_clips = resumen_split["n_clips"].sum()
total_pos = resumen_split["n_ocultamiento"].sum()
total_neg = resumen_split["n_sin_ocultamiento"].sum()

totales_dataset = (
    resumen_split
    .groupby("dataset")["n_clips"]
    .sum()
    .to_dict()
)


def evaluar_asignacion(asignacion):
    """
    Menor score = partición más cercana a las proporciones
    objetivo en tamaño, clase y fuente.
    """

    temp = resumen_split.copy()
    temp["split"] = asignacion

    # ----------------------------------------------------
    # Restricciones mínimas de representación por fuente
    # ----------------------------------------------------

    # CCTV_YOLO tiene cuatro grupos experimentales y puede
    # estar representado en las tres particiones.
    for split in SPLITS:
        tiene_cctv = (
            (temp["split"] == split)
            & (temp["dataset"] == "CCTV_YOLO")
        ).any()

        if not tiene_cctv:
            return np.inf


    # El entrenamiento debe contener ejemplos sintéticos
    # de ambas clases para evitar una asociación directa
    # entre fuente sintética y clase positiva.
    sintetico_train = temp[
        (temp["split"] == "train")
        & (temp["dataset"] == "SINTETICO")
    ]

    if sintetico_train["n_ocultamiento"].sum() == 0:
        return np.inf

    if sintetico_train["n_sin_ocultamiento"].sum() == 0:
        return np.inf

    score = 0.0

    for split in SPLITS:
        prop = PROPORCIONES[split]

        subset = temp[temp["split"] == split]

        # Penalización fuerte si algún split queda vacío
        if len(subset) == 0:
            return np.inf

        # ----------------------------------------------------
        # 1. Tamaño total
        # ----------------------------------------------------

        observado = subset["n_clips"].sum()
        objetivo = total_clips * prop

        score += (
            (observado - objetivo) / objetivo
        ) ** 2

        # ----------------------------------------------------
        # 2. Balance de clases
        # ----------------------------------------------------

        observado_pos = subset["n_ocultamiento"].sum()
        objetivo_pos = total_pos * prop

        observado_neg = subset["n_sin_ocultamiento"].sum()
        objetivo_neg = total_neg * prop

        score += 2.0 * (
            (observado_pos - objetivo_pos)
            / objetivo_pos
        ) ** 2

        score += (
            (observado_neg - objetivo_neg)
            / objetivo_neg
        ) ** 2

        # ----------------------------------------------------
        # 3. Representación por fuente
        # ----------------------------------------------------

        clips_por_dataset = (
            subset
            .groupby("dataset")["n_clips"]
            .sum()
            .to_dict()
        )

        for dataset, total_dataset in totales_dataset.items():

            observado_ds = clips_por_dataset.get(dataset, 0)
            objetivo_ds = total_dataset * prop

            # Se normaliza respecto al tamaño total de la fuente.
            # Esto evita que las fuentes pequeñas dominen el score.
            score += 0.5 * (
                (observado_ds - objetivo_ds)
                / total_dataset
            ) ** 2

    return score


# ------------------------------------------------------------
# Búsqueda reproducible
# ------------------------------------------------------------

mejor_score = np.inf
mejor_asignacion = None

probabilidades = np.array(
    [PROPORCIONES[s] for s in SPLITS]
)

for _ in range(N_INTENTOS):

    asignacion = rng.choice(
        SPLITS,
        size=len(resumen_split),
        p=probabilidades
    )

    score = evaluar_asignacion(asignacion)

    if score < mejor_score:
        mejor_score = score
        mejor_asignacion = asignacion.copy()


resumen_split["split"] = mejor_asignacion

print(f"Mejor score encontrado: {mejor_score:.6f}")

display(
    resumen_split[
        [
            "grupo_experimental",
            "dataset",
            "n_clips",
            "n_ocultamiento",
            "n_sin_ocultamiento",
            "n_videos",
            "split"
        ]
    ]
    .sort_values(
        ["split", "dataset", "grupo_experimental"]
    )
)

Mejor score encontrado: 0.061343


,grupo_experimental,dataset,n_clips,n_ocultamiento,n_sin_ocultamiento,n_videos,split
2,G003,CCTV_YOLO,2,1,1,2,test
6,G007,DCSASS,16,3,13,1,test
13,G014,DCSASS,32,2,30,1,test
21,G022,DCSASS,32,0,32,1,test
26,G027,DCSASS,32,6,26,1,test
...,...,...,...,...,...,...,...
30,G031,DCSASS,32,2,30,1,val
39,G040,MNNIT,17,11,6,17,val
43,G044,MNNIT,13,4,9,13,val
54,G055,SINTETICO,1,1,0,1,val


### 2.4 Validación de las particiones

La función utilizada para construir las particiones selecciona una asignación
de grupos experimentales que aproxima las proporciones objetivo. Sin embargo,
el valor de la función de optimización no constituye por sí mismo evidencia
suficiente de que la partición sea adecuada.

Antes de utilizarla para entrenar el baseline se verifican explícitamente las
siguientes condiciones:

1. Ningún grupo experimental aparece en más de una partición
2. Ninguna persona conocida aparece en más de una partición
3. Entrenamiento, validación y prueba contienen ambas clases
4. Las proporciones de clase se mantienen razonablemente cercanas a la
   distribución global
5. Las diferentes fuentes están representadas en las particiones cuando la
   estructura de los grupos lo permite
6. Las proporciones finales de clips son suficientemente cercanas al objetivo
   de 70/15/15 sin romper las restricciones de independencia.

Estas comprobaciones tienen prioridad sobre obtener porcentajes exactamente
iguales a los objetivos, ya que una división numéricamente más precisa pero
con observaciones relacionadas en diferentes particiones produciría una
estimación optimista del desempeño.

In [8]:
# ------------------------------------------------------------
# Incorporar el split seleccionado a cada clip
# ------------------------------------------------------------

mapa_split = resumen_split.set_index(
    "grupo_experimental"
)["split"]

df_split["split"] = (
    df_split["grupo_experimental"]
    .map(mapa_split)
)

assert df_split["split"].notna().all()

print(
    df_split["split"]
    .value_counts()
)

split
train    702
test     147
val      136
Name: count, dtype: int64


In [9]:
# ------------------------------------------------------------
# Resumen general de las particiones
# ------------------------------------------------------------

resumen_particiones = (
    df_split
    .groupby("split")
    .agg(
        clips=("clip_id", "size"),
        grupos=("grupo_experimental", "nunique"),
        videos=("grupo_video", "nunique"),
        ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        )
    )
    .reindex(["train", "val", "test"])
)

resumen_particiones["porcentaje_clips"] = (
    resumen_particiones["clips"]
    / len(df_split)
    * 100
)

resumen_particiones["pct_ocultamiento"] = (
    resumen_particiones["ocultamiento"]
    / resumen_particiones["clips"]
    * 100
)

resumen_particiones["pct_sin_ocultamiento"] = (
    resumen_particiones["sin_ocultamiento"]
    / resumen_particiones["clips"]
    * 100
)

display(
    resumen_particiones.round(2)
)

,clips,grupos,videos,ocultamiento,sin_ocultamiento,porcentaje_clips,pct_ocultamiento,pct_sin_ocultamiento
split,,,,,,,,
train,702,46,152,132,570,71.27,18.80,81.20
val,136,9,38,30,106,13.81,22.06,77.94
test,147,8,39,28,119,14.92,19.05,80.95


In [10]:
# ------------------------------------------------------------
# Representación de las fuentes
# ------------------------------------------------------------

tabla_fuente_split = pd.crosstab(
    df_split["dataset"],
    df_split["split"]
).reindex(
    columns=["train", "val", "test"],
    fill_value=0
)

display(tabla_fuente_split)

print("\nPorcentaje de cada fuente asignado a cada split:")

tabla_fuente_pct = (
    tabla_fuente_split
    .div(
        tabla_fuente_split.sum(axis=1),
        axis=0
    )
    * 100
)

display(tabla_fuente_pct.round(2))

split,train,val,test
dataset,,,
CCTV_YOLO,4,2,2
DCSASS,569,102,112
MNNIT,120,30,32
SINTETICO,9,2,1



Porcentaje de cada fuente asignado a cada split:


split,train,val,test
dataset,,,
CCTV_YOLO,50.00,25.00,25.00
DCSASS,72.67,13.03,14.30
MNNIT,65.93,16.48,17.58
SINTETICO,75.00,16.67,8.33


In [11]:
# ------------------------------------------------------------
# Distribución conjunta de fuente, clase y split
# ------------------------------------------------------------

tabla_fuente_clase_split = pd.crosstab(
    index=df_split["dataset"],
    columns=[
        df_split["split"],
        df_split["clase_binaria"]
    ]
)

tabla_fuente_clase_split = (
    tabla_fuente_clase_split
    .rename(
        columns={
            0: "sin_ocultamiento",
            1: "ocultamiento"
        },
        level=1
    )
)

display(tabla_fuente_clase_split)

split                     test                         train               \
clase_binaria sin_ocultamiento ocultamiento sin_ocultamiento ocultamiento   
dataset                                                                     
CCTV_YOLO                    1            1                2            2   
DCSASS                     101           11              508           61   
MNNIT                       17           15               58           62   
SINTETICO                    0            1                2            7   

split                      val               
clase_binaria sin_ocultamiento ocultamiento  
dataset                                      
CCTV_YOLO                    1            1  
DCSASS                      90           12  
MNNIT                       15           15  
SINTETICO                    0            2

In [12]:
# ------------------------------------------------------------
# 1. Ningún grupo experimental puede cruzar particiones
# ------------------------------------------------------------

splits_por_grupo = (
    df_split
    .groupby("grupo_experimental")["split"]
    .nunique()
)

assert splits_por_grupo.max() == 1

print(
    "Grupos experimentales presentes en múltiples splits:",
    (splits_por_grupo > 1).sum()
)


# ------------------------------------------------------------
# 2. Ningún video puede cruzar particiones
# ------------------------------------------------------------

splits_por_video = (
    df_split
    .groupby("grupo_video")["split"]
    .nunique()
)

assert splits_por_video.max() == 1

print(
    "Videos presentes en múltiples splits:",
    (splits_por_video > 1).sum()
)


# ------------------------------------------------------------
# 3. Ninguna persona conocida puede cruzar particiones
# ------------------------------------------------------------

personas_validas = df_split[
    df_split["persona_conocida"]
].copy()

personas_validas["grupo_persona"] = (
    personas_validas["dataset"].astype(str)
    + "__"
    + personas_validas["persona_id_limpio"]
)

splits_por_persona = (
    personas_validas
    .groupby("grupo_persona")["split"]
    .nunique()
)

assert splits_por_persona.max() == 1

print(
    "Personas conocidas presentes en múltiples splits:",
    (splits_por_persona > 1).sum()
)


# ------------------------------------------------------------
# 4. Las tres particiones deben contener ambas clases
# ------------------------------------------------------------

clases_por_split = (
    df_split
    .groupby("split")["clase_binaria"]
    .nunique()
)

assert (clases_por_split == 2).all()

print("\nClases diferentes por split:")
display(clases_por_split.to_frame("n_clases"))

Grupos experimentales presentes en múltiples splits: 0
Videos presentes en múltiples splits: 0
Personas conocidas presentes en múltiples splits: 0

Clases diferentes por split:


,n_clases
split,
test,2
train,2
val,2


### 2.4.1 Resultado de la validación de las particiones

La partición seleccionada contiene 702 clips de entrenamiento (71.27 %),
136 de validación (13.81 %) y 147 de prueba (14.92 %). Las diferencias respecto
al objetivo de 70/15/15 se consideran aceptables debido a que la asignación
se realiza sobre grupos experimentales indivisibles y no sobre clips
independientes.

La distribución de la clase `ocultamiento` es de 18.80 % en entrenamiento,
22.06 % en validación y 19.05 % en prueba, frente a 19.29 % en el conjunto
elegible completo. En particular, la proporción observada en prueba es cercana
a la prevalencia global del dataset.

Las cuatro fuentes se encuentran representadas en entrenamiento, validación
y prueba. CCTV_YOLO conserva ejemplos de ambas clases en las tres particiones.
DCSASS mantiene una proporción de `ocultamiento` cercana al 10-12 % en los
tres conjuntos, mientras que MNNIT permanece aproximadamente balanceado entre
las dos clases.

SINTETICO constituye una excepción debido a su reducido tamaño y distribución
original de 10 clips de `ocultamiento` y solamente 2 de `sin_ocultamiento`.
Los dos ejemplos negativos se mantienen en entrenamiento, de manera que el
modelo no observe exclusivamente ejemplos positivos de esta fuente durante
el ajuste. Como consecuencia, los subconjuntos sintéticos de validación y
prueba contienen únicamente ejemplos positivos y las métricas por fuente que
requieren presencia de ambas clases no se calcularán para esos subconjuntos.

Las comprobaciones de independencia confirmaron que ningún grupo experimental,
video de origen ni persona conocida aparece en más de una partición. Las tres
particiones contienen además ejemplos de ambas clases.

Con base en estas verificaciones, la partición se considera adecuada para el
experimento baseline y se mantendrá fija para las arquitecturas comparadas
posteriormente.

In [13]:
# ------------------------------------------------------------
# Persistencia de la partición experimental
# ------------------------------------------------------------

SPLIT_PATH = BASE_DIR / "baseline_split_equipo41.csv"

columnas_split = [
    "dataset",
    "video_id",
    "video_origen",
    "clip_id",
    "nombre_clip",
    "persona_id",
    "clase_binaria",
    "grupo_video",
    "grupo_experimental",
    "split"
]

df_split_final = (
    df_split[columnas_split]
    .copy()
    .sort_values(
        ["split", "dataset", "grupo_experimental", "clip_id"]
    )
    .reset_index(drop=True)
)

df_split_final.to_csv(
    SPLIT_PATH,
    index=False
)

print("Archivo guardado:")
print(SPLIT_PATH)

print(f"\nRegistros guardados: {len(df_split_final):,}")

print("\nDistribución:")
print(
    df_split_final["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

assert len(df_split_final) == 985
assert df_split_final["split"].notna().all()
assert df_split_final["clip_id"].notna().all()

Archivo guardado:
/mnt/c/dev/MNA---Proyecto-Integrador/Datos/Preparacion_Datasets/baseline_split_equipo41.csv

Registros guardados: 985

Distribución:
split
train    702
val      136
test     147
Name: count, dtype: int64


In [14]:
import hashlib

with open(SPLIT_PATH, "rb") as f:
    split_sha256 = hashlib.sha256(
        f.read()
    ).hexdigest()

print("SHA-256 del split:")
print(split_sha256)

SHA-256 del split:
7ed380a00a883c42308670842cbb554881745a455fd4b54cb5fa0ec3ad6d994b


## 3. Selección del algoritmo baseline

El modelo seleccionado como baseline es **R3D-18**, una red neuronal
convolucional tridimensional diseñada para procesar secuencias de video.
A diferencia de una arquitectura 2D aplicada de forma independiente a cada
frame, las convoluciones 3D operan simultáneamente sobre las dimensiones
espaciales y temporal de la entrada, permitiendo aprender patrones de
apariencia y movimiento.

Esta característica es pertinente para el problema del proyecto porque la
distinción entre `ocultamiento` y `sin_ocultamiento` depende de una secuencia
de acciones y no únicamente de la apariencia de un frame aislado. El modelo
recibe una representación con forma `(C, T, H, W)`, donde `T = 16` corresponde
a los frames seleccionados uniformemente a lo largo de cada clip.

R3D-18 se utiliza con pesos preentrenados en Kinetics-400 y posteriormente se
adapta mediante transferencia de aprendizaje a las dos clases del proyecto.
El preentrenamiento proporciona representaciones previamente aprendidas sobre
acciones humanas y permite utilizar el conjunto disponible para ajustar el
modelo al dominio específico sin iniciar el aprendizaje completamente desde
pesos aleatorios.

La arquitectura se considera apropiada como baseline por las siguientes
razones:

- Procesa directamente información espacio-temporal RGB
- Corresponde a una arquitectura conocida y reproducible para reconocimiento
  de acciones
- Su complejidad es moderada respecto a arquitecturas de video más recientes
  y de mayor tamaño
- Permite utilizar transferencia de aprendizaje con Kinetics-400
- Puede entrenarse y evaluarse con los recursos computacionales disponibles
  para el proyecto
- Produce una referencia suficientemente sólida para comparar posteriormente
  arquitecturas alternativas

La principal limitación de interpretabilidad es que las variables utilizadas
para la decisión no corresponden a características definidas manualmente. Las
representaciones se aprenden internamente mediante las capas convolucionales.
Por esta razón, posteriormente se analizarán los embeddings de la penúltima
etapa de la red y el comportamiento de los errores de clasificación.

El baseline no pretende representar la arquitectura final de una solución
productiva. Su función es establecer un desempeño de referencia reproducible
contra el cual puedan compararse modelos posteriores utilizando las mismas
particiones experimentales.

### 3.1 Entorno de ejecución y reproducibilidad

Antes del entrenamiento se documentan las versiones de las principales
bibliotecas, el dispositivo disponible y la configuración utilizada para la
reproducibilidad.

El registro del entorno permite distinguir cambios derivados del modelo de
posibles diferencias introducidas por versiones de PyTorch, Torchvision,
CUDA u otras dependencias. También permite conservar evidencia de los recursos
computacionales utilizados durante el experimento.

In [15]:
import sys
import platform

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())

print("\nBibliotecas")
print("-" * 40)
print("NumPy:      ", np.__version__)
print("Pandas:     ", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:    ", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("\nDispositivo")
print("-" * 40)

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA PyTorch:", torch.version.cuda)

    propiedades_gpu = torch.cuda.get_device_properties(0)

    memoria_gb = (
        propiedades_gpu.total_memory
        / 1024**3
    )

    print(
        "Memoria GPU:",
        f"{memoria_gb:.2f} GB"
    )

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDispositivo seleccionado:", device)

Python: 3.12.3
Sistema: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39

Bibliotecas
----------------------------------------
NumPy:       2.5.2
Pandas:      3.0.6
Scikit-learn: 1.9.1
PyTorch:     2.14.1+cu126
Torchvision: 0.29.1+cu126

Dispositivo
----------------------------------------
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA PyTorch: 12.6
Memoria GPU: 6.00 GB

Dispositivo seleccionado: cuda


In [16]:
import random

SEED_MODEL = 41

random.seed(SEED_MODEL)
np.random.seed(SEED_MODEL)
torch.manual_seed(SEED_MODEL)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED_MODEL)
    torch.cuda.manual_seed_all(SEED_MODEL)

print("Semilla del experimento:", SEED_MODEL)

Semilla del experimento: 41


In [17]:
import os
from pathlib import Path

import cv2

print("=== Dispositivo ===")
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== Dataset remoto ===")

bnext_path = os.environ.get("BNEXT_CLIPS_DIR")

print("BNEXT_CLIPS_DIR:", bnext_path)

if bnext_path is None:
    raise RuntimeError(
        "La variable de entorno BNEXT_CLIPS_DIR no está definida."
    )

CLIPS_DIR = Path(bnext_path)

print("Ruta disponible:", CLIPS_DIR.exists())

archivos_mp4 = sorted(CLIPS_DIR.glob("*.mp4"))

print("Archivos MP4 visibles:", len(archivos_mp4))

if not archivos_mp4:
    raise RuntimeError(
        "No se encontraron archivos MP4 en BNEXT_CLIPS_DIR."
    )

video_prueba = archivos_mp4[0]

print("Video de prueba:", video_prueba.name)

cap = cv2.VideoCapture(str(video_prueba))

if not cap.isOpened():
    raise RuntimeError(
        f"No se pudo abrir el video remoto: {video_prueba}"
    )

ret, frame = cap.read()

fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print("\n=== Lectura del video ===")
print("Primer frame leído:", ret)
print("Forma del frame:", frame.shape if ret else None)
print("FPS:", fps)
print("Frames:", n_frames)
print("Resolución:", f"{width} × {height}")

assert ret, "OpenCV no pudo leer el primer frame."

=== Dispositivo ===
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU

=== Dataset remoto ===
BNEXT_CLIPS_DIR: /home/ajimenez/mnt/bnext_clips
Ruta disponible: True
Archivos MP4 visibles: 987
Video de prueba: CCTV_YOLO__CCTV_001__CCTV_001.mp4

=== Lectura del video ===
Primer frame leído: True
Forma del frame: (544, 544, 3)
FPS: 24.0
Frames: 96
Resolución: 544 × 544


### 3.2 Inicialización de R3D-18 mediante transferencia de aprendizaje

El baseline utiliza R3D-18 con pesos preentrenados en Kinetics-400.
La arquitectura original fue diseñada para clasificar 400 categorías de
acciones en video, por lo que su capa final debe adaptarse al problema binario
del proyecto.

Se conserva el extractor espacio-temporal preentrenado y se sustituye la capa
de clasificación final por una capa lineal con dos salidas:

- `0`: `sin_ocultamiento`
- `1`: `ocultamiento`

La entrada de la capa final de R3D-18 tiene 512 características. Estas
representaciones internas también se utilizarán posteriormente para analizar
las características aprendidas por el baseline.

En esta etapa únicamente se inicializa y verifica la arquitectura. La
estrategia de ajuste de los parámetros, la función de pérdida y los
hiperparámetros se definirán después de comprobar el funcionamiento del
modelo con los datos preparados.

In [18]:
import torch.nn as nn

from torchvision.models.video import (
    r3d_18,
    R3D_18_Weights
)

# ------------------------------------------------------------
# Inicialización del modelo preentrenado
# ------------------------------------------------------------

weights = R3D_18_Weights.KINETICS400_V1

model = r3d_18(
    weights=weights
)

print("Capa final original:")
print(model.fc)


# ------------------------------------------------------------
# Adaptación a clasificación binaria
# ------------------------------------------------------------

n_features = model.fc.in_features

model.fc = nn.Linear(
    in_features=n_features,
    out_features=2
)

model = model.to(device)

print("\nCaracterísticas antes del clasificador:", n_features)

print("\nCapa final modificada:")
print(model.fc)

print("\nDispositivo del modelo:")
print(next(model.parameters()).device)


# ------------------------------------------------------------
# Número de parámetros
# ------------------------------------------------------------

n_parametros = sum(
    p.numel()
    for p in model.parameters()
)

n_entrenables = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"\nParámetros totales: "
    f"{n_parametros:,}"
)

print(
    f"Parámetros entrenables actualmente: "
    f"{n_entrenables:,}"
)

Capa final original:
Linear(in_features=512, out_features=400, bias=True)

Características antes del clasificador: 512

Capa final modificada:
Linear(in_features=512, out_features=2, bias=True)

Dispositivo del modelo:
cuda:0

Parámetros totales: 33,167,298
Parámetros entrenables actualmente: 33,167,298


### 3.3 Estrategia de transferencia de aprendizaje

Aunque R3D-18 contiene aproximadamente 33 millones de parámetros, el conjunto
de entrenamiento disponible para este experimento contiene 702 clips. Realizar
un ajuste completo de toda la arquitectura desde el inicio incrementaría el
costo computacional y el riesgo de sobreajuste.

Por otra parte, entrenar únicamente la capa final aprovecharía las
representaciones de Kinetics-400 sin permitir que el extractor espacio-temporal
se adapte al dominio de videos de retail.

Para el baseline se adopta una estrategia intermedia de transferencia de
aprendizaje. Se congelan el `stem` y los bloques `layer1`, `layer2` y `layer3`,
mientras que `layer4` y la nueva capa `fc` permanecen entrenables.

Las primeras etapas de la red conservan así las representaciones
espacio-temporales generales aprendidas durante el preentrenamiento, mientras
que las capas de mayor nivel pueden adaptarse a los patrones relevantes para
distinguir `ocultamiento` y `sin_ocultamiento`.

Esta configuración también reduce el número de parámetros que requieren
actualización y constituye una estrategia reproducible para establecer el
baseline. El desempeño obtenido permitirá determinar posteriormente si modelos
o estrategias de ajuste más complejas producen una mejora relevante.

In [19]:
# ------------------------------------------------------------
# Congelar el extractor de características inicial
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# Mantener entrenables layer4 y fc
# ------------------------------------------------------------

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Conteo de parámetros
# ------------------------------------------------------------

n_parametros = sum(
    p.numel()
    for p in model.parameters()
)

n_entrenables = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

n_congelados = n_parametros - n_entrenables

print(f"Parámetros totales:     {n_parametros:,}")
print(f"Parámetros entrenables: {n_entrenables:,}")
print(f"Parámetros congelados:  {n_congelados:,}")

print(
    "Porcentaje entrenable:",
    f"{100 * n_entrenables / n_parametros:.2f}%"
)


# ------------------------------------------------------------
# Verificar qué bloques quedan entrenables
# ------------------------------------------------------------

print("\nEstado por bloque:")
print("-" * 50)

for nombre_bloque in [
    "stem",
    "layer1",
    "layer2",
    "layer3",
    "layer4",
    "fc"
]:
    bloque = getattr(model, nombre_bloque)

    total = sum(
        p.numel()
        for p in bloque.parameters()
    )

    entrenables = sum(
        p.numel()
        for p in bloque.parameters()
        if p.requires_grad
    )

    print(
        f"{nombre_bloque:8s} "
        f"{entrenables:>12,} / {total:>12,} "
        f"entrenables"
    )

Parámetros totales:     33,167,298
Parámetros entrenables: 24,909,826
Parámetros congelados:  8,257,472
Porcentaje entrenable: 75.10%

Estado por bloque:
--------------------------------------------------
stem                0 /       28,352 entrenables
layer1              0 /      442,880 entrenables
layer2              0 /    1,557,760 entrenables
layer3              0 /    6,228,480 entrenables
layer4     24,908,800 /   24,908,800 entrenables
fc              1,026 /        1,026 entrenables


#### 3.3.1 Evaluación de la configuración inicial de fine-tuning

La primera configuración propuesta mantenía entrenables `layer4` y `fc`.
Sin embargo, al inspeccionar el número de parámetros de R3D-18 se observó que
`layer4` concentra aproximadamente el 75 % de los parámetros totales del
modelo:

- parámetros totales: 33,167,298;
- parámetros entrenables con `layer4 + fc`: 24,909,826;
- porcentaje entrenable: 75.10 %.

Esto implica que, aunque las primeras capas permanecieran congeladas, la
configuración seguiría actualizando la mayor parte de la arquitectura. Para un
conjunto de entrenamiento de 702 clips, esta proporción resulta elevada para
el propósito de establecer un baseline mediante transferencia de aprendizaje,
ya que incrementa el costo computacional y el riesgo de sobreajuste.

Por esta razón se adopta una configuración más conservadora. Se mantienen
congelados `stem`, `layer1`, `layer2`, `layer3` y el primer bloque residual
`layer4[0]`. Únicamente se ajustan el último bloque residual `layer4[1]` y la
nueva capa de clasificación `fc`.

Esta decisión conserva la mayor parte de las representaciones espacio-temporales
aprendidas durante el preentrenamiento en Kinetics-400, pero permite adaptar
las características de mayor nivel al dominio específico de los videos de
retail.

In [20]:
# ------------------------------------------------------------
# Congelar inicialmente toda la arquitectura
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# Descongelar únicamente el último bloque residual de layer4
# y la nueva capa clasificadora
# ------------------------------------------------------------

for param in model.layer4[1].parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Conteo de parámetros
# ------------------------------------------------------------

n_parametros = sum(
    p.numel()
    for p in model.parameters()
)

n_entrenables = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

n_congelados = n_parametros - n_entrenables

print(f"Parámetros totales:     {n_parametros:,}")
print(f"Parámetros entrenables: {n_entrenables:,}")
print(f"Parámetros congelados:  {n_congelados:,}")

print(
    "Porcentaje entrenable:",
    f"{100 * n_entrenables / n_parametros:.2f}%"
)


# ------------------------------------------------------------
# Estado de los bloques principales
# ------------------------------------------------------------

print("\nEstado por bloque:")
print("-" * 55)

bloques = {
    "stem": model.stem,
    "layer1": model.layer1,
    "layer2": model.layer2,
    "layer3": model.layer3,
    "layer4[0]": model.layer4[0],
    "layer4[1]": model.layer4[1],
    "fc": model.fc,
}

for nombre, bloque in bloques.items():

    total = sum(
        p.numel()
        for p in bloque.parameters()
    )

    entrenables = sum(
        p.numel()
        for p in bloque.parameters()
        if p.requires_grad
    )

    print(
        f"{nombre:10s} "
        f"{entrenables:>12,} / "
        f"{total:>12,} entrenables"
    )

Parámetros totales:     33,167,298
Parámetros entrenables: 14,158,850
Parámetros congelados:  19,008,448
Porcentaje entrenable: 42.69%

Estado por bloque:
-------------------------------------------------------
stem                  0 /       28,352 entrenables
layer1                0 /      442,880 entrenables
layer2                0 /    1,557,760 entrenables
layer3                0 /    6,228,480 entrenables
layer4[0]             0 /   10,750,976 entrenables
layer4[1]    14,157,824 /   14,157,824 entrenables
fc                1,026 /        1,026 entrenables


#### 3.3.2 Configuración final de fine-tuning

Después del ajuste, el modelo mantiene entrenables 14,158,850 parámetros,
equivalentes al 42.69 % de los 33,167,298 parámetros totales de R3D-18.

La configuración final conserva congelados `stem`, `layer1`, `layer2`,
`layer3` y `layer4[0]`. Se mantienen entrenables únicamente `layer4[1]`
y la nueva capa `fc`.

Esta configuración reduce de 75.10 % a 42.69 % la proporción de parámetros
actualizados respecto a la propuesta inicial. Con ello se conserva una mayor
parte de las representaciones aprendidas en Kinetics-400 y se limita la
capacidad de adaptación a las capas de mayor nivel, donde se espera que las
representaciones sean más específicas de las acciones observadas.

La estrategia representa un compromiso entre dos extremos: entrenar solamente
el nuevo clasificador, lo que limitaría la adaptación al dominio de retail, y
realizar un fine-tuning completo, que incrementaría el costo computacional y
la capacidad de sobreajuste sobre un conjunto de entrenamiento relativamente
pequeño.

Esta configuración se utilizará como estrategia de transferencia de
aprendizaje del baseline y permanecerá fija durante su entrenamiento.

### 3.4 Preparación del Dataset para R3D-18

La preparación de cada clip reutiliza el procedimiento definido y validado en
el Avance 2. No se introduce una nueva transformación temporal o espacial para
el baseline, con el objetivo de mantener continuidad entre la etapa de
preparación de datos y la etapa de modelado.

Para cada video se realizan las siguientes operaciones:

1. se obtienen 16 frames distribuidos uniformemente a lo largo de la duración
   completa del clip;
2. cada frame se convierte de BGR a RGB;
3. se redimensiona el frame completo a 112 × 112 píxeles;
4. los valores se convierten a `float32` y se escalan al intervalo [0, 1];
5. se aplica la normalización correspondiente a los pesos preentrenados de
   R3D-18 sobre Kinetics-400;
6. los frames se reorganizan en un tensor con forma `(C, T, H, W)`.

La selección temporal uniforme permite representar clips con distintas
duraciones mediante una entrada de tamaño fijo sin imponer una duración física
de cuatro segundos. El conjunto utilizado en este avance contiene clips con
diferentes duraciones, pero todos disponen de suficientes frames para construir
la representación de 16 frames.

La clase `Dataset` utiliza la partición experimental previamente congelada.
Por tanto, cada observación mantiene la asignación a `train`, `val` o `test`
definida antes del entrenamiento.

In [21]:
import cv2
import numpy as np
import torch

# ------------------------------------------------------------
# Configuración heredada del Avance 2
# ------------------------------------------------------------

NUM_FRAMES = 16
FRAME_SIZE = 112

KINETICS_MEAN = np.array(
    [0.43216, 0.394666, 0.37645],
    dtype=np.float32
)

KINETICS_STD = np.array(
    [0.22803, 0.22145, 0.216989],
    dtype=np.float32
)


def cargar_clip_r3d(video_path):
    """
    Lee un video y genera el tensor de entrada para R3D-18.

    Salida:
        torch.Tensor con forma (C, T, H, W)
    """

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"No se pudo abrir el video: {video_path}"
        )

    n_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if n_frames < NUM_FRAMES:
        cap.release()

        raise RuntimeError(
            f"El video {video_path} contiene "
            f"{n_frames} frames, menos de los "
            f"{NUM_FRAMES} requeridos."
        )

    # Índices distribuidos uniformemente sobre
    # la duración completa del clip
    indices = np.linspace(
        0,
        n_frames - 1,
        NUM_FRAMES
    )

    indices = np.round(indices).astype(int)

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        ret, frame = cap.read()

        if not ret:
            cap.release()

            raise RuntimeError(
                f"No se pudo leer el frame {idx} "
                f"de {video_path}"
            )

        # BGR -> RGB
        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Resize del frame completo
        frame = cv2.resize(
            frame,
            (FRAME_SIZE, FRAME_SIZE),
            interpolation=cv2.INTER_LINEAR
        )

        # [0, 255] -> [0, 1]
        frame = (
            frame.astype(np.float32)
            / 255.0
        )

        # Normalización Kinetics-400
        frame = (
            frame - KINETICS_MEAN
        ) / KINETICS_STD

        frames.append(frame)

    cap.release()

    # (T, H, W, C)
    clip = np.stack(
        frames,
        axis=0
    )

    # (C, T, H, W)
    clip = np.transpose(
        clip,
        (3, 0, 1, 2)
    )

    return torch.from_numpy(
        clip
    ).float()

In [22]:
video_prueba = archivos_mp4[0]

clip_prueba = cargar_clip_r3d(
    video_prueba
)

print("Video:", video_prueba.name)
print("Forma:", tuple(clip_prueba.shape))
print("Tipo:", clip_prueba.dtype)

print(
    "Valores finitos:",
    torch.isfinite(
        clip_prueba
    ).all().item()
)

print(
    "Mínimo:",
    clip_prueba.min().item()
)

print(
    "Máximo:",
    clip_prueba.max().item()
)

print(
    "Media:",
    clip_prueba.mean().item()
)

Video: CCTV_YOLO__CCTV_001__CCTV_001.mp4
Forma: (3, 16, 112, 112)
Tipo: torch.float32
Valores finitos: True
Mínimo: -1.8951891660690308
Máximo: 2.494122266769409
Media: 0.45742693543434143


#### 3.4.1 Dataset de clips y etiquetas

La clase utilizada por PyTorch conecta dos artefactos definidos previamente:

- La partición experimental congelada, que contiene la asignación de cada clip
  a `train`, `val` o `test`
- El directorio remoto de videos, accesible mediante `BNEXT_CLIPS_DIR`

Cada elemento del dataset devuelve el tensor preprocesado del clip y su clase
binaria. La preparación temporal y espacial se realiza en el momento de cargar
el ejemplo, por lo que no se generan copias locales de los videos ni archivos
intermedios de frames.

La clase no modifica las particiones ni las etiquetas. Su función es únicamente
traducir cada registro de la tabla experimental a la representación requerida
por R3D-18.

In [23]:
from torch.utils.data import Dataset


class VideoDataset(Dataset):
    """
    Dataset para clasificación binaria de clips con R3D-18.
    """

    def __init__(
        self,
        dataframe,
        clips_dir
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.clips_dir = Path(clips_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        fila = self.df.iloc[idx]

        video_path = (
            self.clips_dir
            / fila["nombre_clip"]
        )

        if not video_path.exists():
            raise FileNotFoundError(
                f"No se encontró el clip: {video_path}"
            )

        clip = cargar_clip_r3d(
            video_path
        )

        etiqueta = torch.tensor(
            int(fila["clase_binaria"]),
            dtype=torch.long
        )

        return clip, etiqueta

In [24]:
df_train = (
    df_split_final[
        df_split_final["split"] == "train"
    ]
    .reset_index(drop=True)
)

df_val = (
    df_split_final[
        df_split_final["split"] == "val"
    ]
    .reset_index(drop=True)
)

df_test = (
    df_split_final[
        df_split_final["split"] == "test"
    ]
    .reset_index(drop=True)
)


train_dataset = VideoDataset(
    df_train,
    CLIPS_DIR
)

val_dataset = VideoDataset(
    df_val,
    CLIPS_DIR
)

test_dataset = VideoDataset(
    df_test,
    CLIPS_DIR
)


print("Elementos por dataset")
print("-" * 35)
print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))

Elementos por dataset
-----------------------------------
Train: 702
Val:   136
Test:  147


In [25]:
for nombre, dataset in [
    ("train", train_dataset),
    ("val", val_dataset),
    ("test", test_dataset),
]:

    clip, etiqueta = dataset[0]

    print(f"\n{nombre.upper()}")
    print("-" * 35)

    print(
        "Forma:",
        tuple(clip.shape)
    )

    print(
        "Tipo tensor:",
        clip.dtype
    )

    print(
        "Etiqueta:",
        etiqueta.item()
    )

    print(
        "Tipo etiqueta:",
        etiqueta.dtype
    )

    print(
        "Valores finitos:",
        torch.isfinite(clip).all().item()
    )


TRAIN
-----------------------------------
Forma: (3, 16, 112, 112)
Tipo tensor: torch.float32
Etiqueta: 0
Tipo etiqueta: torch.int64
Valores finitos: True

VAL
-----------------------------------
Forma: (3, 16, 112, 112)
Tipo tensor: torch.float32
Etiqueta: 0
Tipo etiqueta: torch.int64
Valores finitos: True

TEST
-----------------------------------
Forma: (3, 16, 112, 112)
Tipo tensor: torch.float32
Etiqueta: 0
Tipo etiqueta: torch.int64
Valores finitos: True


#### 3.4.2 Construcción de DataLoaders

Los conjuntos de entrenamiento, validación y prueba se integran con
`DataLoader` para formar lotes con estructura `(B, C, T, H, W)`.

El conjunto de entrenamiento se mezcla en cada época, mientras que validación
y prueba mantienen un orden fijo. La mezcla de los clips de entrenamiento no
modifica la partición experimental ni las relaciones entre grupos, ya que
todos los ejemplos permanecen dentro del conjunto al que fueron asignados.

Debido a que el equipo utilizado dispone de 6 GB de memoria GPU, el tamaño de
lote se determina mediante una prueba empírica antes del entrenamiento. Se
comienza con un valor conservador y se verifica que una pasada hacia adelante
y hacia atrás pueda ejecutarse sin agotar la memoria disponible.

In [26]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4
NUM_WORKERS = 0

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED_MODEL)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Batches")
print("-" * 30)
print("Train:", len(train_loader))
print("Val:  ", len(val_loader))
print("Test: ", len(test_loader))

Batches
------------------------------
Train: 176
Val:   34
Test:  37


In [27]:
clips_batch, etiquetas_batch = next(iter(train_loader))

print("Forma del batch:", tuple(clips_batch.shape))
print("Forma etiquetas:", tuple(etiquetas_batch.shape))
print("Tipo clips:", clips_batch.dtype)
print("Tipo etiquetas:", etiquetas_batch.dtype)
print("Etiquetas:", etiquetas_batch.tolist())

Forma del batch: (4, 3, 16, 112, 112)
Forma etiquetas: (4,)
Tipo clips: torch.float32
Tipo etiquetas: torch.int64
Etiquetas: [0, 0, 0, 1]


In [28]:
import torch.nn.functional as F

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()

clips_batch = clips_batch.to(
    device,
    non_blocking=True
)

etiquetas_batch = etiquetas_batch.to(
    device,
    non_blocking=True
)

# Gradientes limpios
model.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(clips_batch)

    loss_prueba = F.cross_entropy(
        logits,
        etiquetas_batch
    )

loss_prueba.backward()

memoria_asignada = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

memoria_reservada = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)

print("Forma de salida:", tuple(logits.shape))
print("Loss de prueba:", loss_prueba.item())

print(
    "Memoria máxima asignada:",
    f"{memoria_asignada:.2f} GB"
)

print(
    "Memoria máxima reservada:",
    f"{memoria_reservada:.2f} GB"
)

model.zero_grad(set_to_none=True)

del clips_batch
del etiquetas_batch
del logits
del loss_prueba

torch.cuda.empty_cache()

Forma de salida: (4, 2)
Loss de prueba: 0.6534423828125
Memoria máxima asignada: 0.28 GB
Memoria máxima reservada: 0.31 GB


#### 3.3.3 Control de los bloques congelados durante entrenamiento

Congelar un parámetro mediante `requires_grad = False` evita que sus pesos se
actualicen mediante retropropagación, pero no cambia automáticamente el modo de
operación de las capas de normalización.

R3D-18 contiene capas de Batch Normalization que mantienen estadísticas
internas de activación. Si toda la arquitectura se coloca en modo de
entrenamiento, estas estadísticas podrían continuar modificándose incluso en
los bloques cuyos parámetros se encuentran congelados.

Para mantener estable la parte preentrenada del modelo, durante cada época se
coloca la arquitectura en modo de entrenamiento y posteriormente se mantienen
en modo de evaluación los bloques congelados. `layer4[1]` y `fc` permanecen
en modo de entrenamiento.

Esta configuración permite actualizar únicamente la parte seleccionada para
fine-tuning y conservar tanto los pesos como las estadísticas internas de las
capas congeladas.

In [29]:
def configurar_modo_entrenamiento(model):
    """
    Activa entrenamiento únicamente en los bloques utilizados
    para fine-tuning y mantiene en evaluación los bloques congelados.
    """

    model.train()

    model.stem.eval()
    model.layer1.eval()
    model.layer2.eval()
    model.layer3.eval()
    model.layer4[0].eval()

    model.layer4[1].train()
    model.fc.train()

In [30]:
BATCH_SIZE_PRUEBA = 8

loader_prueba = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_PRUEBA,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=loader_generator
)

clips_batch, etiquetas_batch = next(iter(loader_prueba))

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

configurar_modo_entrenamiento(model)

clips_batch = clips_batch.to(
    device,
    non_blocking=True
)

etiquetas_batch = etiquetas_batch.to(
    device,
    non_blocking=True
)

model.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(clips_batch)

    loss_prueba = F.cross_entropy(
        logits,
        etiquetas_batch
    )

loss_prueba.backward()

memoria_asignada = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

memoria_reservada = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)

print("Batch probado:", BATCH_SIZE_PRUEBA)
print("Forma de entrada:", tuple(clips_batch.shape))
print("Forma de salida:", tuple(logits.shape))
print("Loss de prueba:", loss_prueba.item())

print(
    "Memoria máxima asignada:",
    f"{memoria_asignada:.2f} GB"
)

print(
    "Memoria máxima reservada:",
    f"{memoria_reservada:.2f} GB"
)

model.zero_grad(set_to_none=True)

del clips_batch
del etiquetas_batch
del logits
del loss_prueba

torch.cuda.empty_cache()

Batch probado: 8
Forma de entrada: (8, 3, 16, 112, 112)
Forma de salida: (8, 2)
Loss de prueba: 0.725616455078125
Memoria máxima asignada: 0.45 GB
Memoria máxima reservada: 0.51 GB


In [31]:
BATCH_SIZE_PRUEBA = 16

loader_prueba = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_PRUEBA,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=loader_generator
)

clips_batch, etiquetas_batch = next(iter(loader_prueba))

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

configurar_modo_entrenamiento(model)

clips_batch = clips_batch.to(
    device,
    non_blocking=True
)

etiquetas_batch = etiquetas_batch.to(
    device,
    non_blocking=True
)

model.zero_grad(set_to_none=True)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(clips_batch)

    loss_prueba = F.cross_entropy(
        logits,
        etiquetas_batch
    )

loss_prueba.backward()

memoria_asignada = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

memoria_reservada = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)

print("Batch probado:", BATCH_SIZE_PRUEBA)
print("Forma de entrada:", tuple(clips_batch.shape))
print("Forma de salida:", tuple(logits.shape))
print("Loss de prueba:", loss_prueba.item())

print(
    "Memoria máxima asignada:",
    f"{memoria_asignada:.2f} GB"
)

print(
    "Memoria máxima reservada:",
    f"{memoria_reservada:.2f} GB"
)

model.zero_grad(set_to_none=True)

del clips_batch
del etiquetas_batch
del logits
del loss_prueba

torch.cuda.empty_cache()

Batch probado: 16
Forma de entrada: (16, 3, 16, 112, 112)
Forma de salida: (16, 2)
Loss de prueba: 0.6496429443359375
Memoria máxima asignada: 0.75 GB
Memoria máxima reservada: 0.84 GB


#### 3.4.3 Selección del tamaño de lote

El tamaño de lote se determinó empíricamente considerando la memoria disponible
en la NVIDIA GeForce RTX 3050 de 6 GB y la configuración de fine-tuning
seleccionada.

Se realizaron pruebas de pasada hacia adelante y retropropagación utilizando
precisión mixta con tamaños de lote de 4, 8 y 16. Los consumos máximos de
memoria reservada fueron aproximadamente 0.31 GB, 0.51 GB y 0.84 GB,
respectivamente.

Los tres tamaños pudieron procesarse sin agotar la memoria disponible. Para el
baseline se selecciona `batch_size = 16`, ya que proporciona margen suficiente
para los componentes adicionales del entrenamiento, como los estados del
optimizador, y reduce el número de lotes procesados por época.

Con 702 clips de entrenamiento, este tamaño produce aproximadamente 44 lotes
por época. No se busca utilizar el tamaño máximo que admite físicamente la GPU,
sino establecer una configuración estable y reproducible para el experimento.

In [32]:
# ------------------------------------------------------------
# Configuración final de DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 16
NUM_WORKERS = 0

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED_MODEL)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Configuración final")
print("-" * 40)
print("Batch size:  ", BATCH_SIZE)
print("Num workers: ", NUM_WORKERS)

print("\nNúmero de batches")
print("-" * 40)
print("Train:", len(train_loader))
print("Val:  ", len(val_loader))
print("Test: ", len(test_loader))

Configuración final
----------------------------------------
Batch size:   16
Num workers:  0

Número de batches
----------------------------------------
Train: 44
Val:   9
Test:  10


### 3.5 Función de pérdida y tratamiento del desbalance

El conjunto de entrenamiento presenta un desbalance entre las dos clases. De
los 702 clips utilizados para ajustar el modelo, 570 corresponden a
`sin_ocultamiento` y 132 a `ocultamiento`. La clase positiva representa
aproximadamente el 18.8 % del conjunto de entrenamiento.

Utilizar una función de pérdida sin ningún ajuste permitiría que la clase
mayoritaria tuviera una influencia considerablemente mayor durante la
optimización. Para reducir este efecto se utiliza `CrossEntropyLoss` con pesos
de clase calculados exclusivamente a partir de la distribución del conjunto de
entrenamiento.

Los pesos se calculan mediante la expresión:

$$
w_c = \frac{N}{K\,n_c}
$$

donde:

- \(N\) es el número total de ejemplos de entrenamiento;
- \(K\) es el número de clases;
- \(n_c\) es el número de ejemplos pertenecientes a la clase \(c\).

De esta manera, los errores cometidos sobre la clase minoritaria
`ocultamiento` reciben un peso mayor durante la actualización de los
parámetros.

El ajuste se aplica únicamente a la función de pérdida durante entrenamiento.
Las distribuciones de validación y prueba permanecen sin modificación para
que las métricas reflejen el comportamiento del modelo sobre los datos
originales.

No se combina esta estrategia con sobremuestreo o submuestreo en el baseline,
con el objetivo de evitar introducir simultáneamente varios mecanismos de
balanceo y mantener una configuración experimental interpretable.

In [33]:
import torch.nn as nn

# ------------------------------------------------------------
# Distribución de clases en entrenamiento
# ------------------------------------------------------------

conteos_train = (
    df_train["clase_binaria"]
    .value_counts()
    .sort_index()
)

n_clase_0 = conteos_train.loc[0]
n_clase_1 = conteos_train.loc[1]

n_total = len(df_train)
n_clases = 2

# ------------------------------------------------------------
# Pesos balanceados
# w_c = N / (K * n_c)
# ------------------------------------------------------------

peso_0 = n_total / (n_clases * n_clase_0)
peso_1 = n_total / (n_clases * n_clase_1)

class_weights = torch.tensor(
    [peso_0, peso_1],
    dtype=torch.float32,
    device=device
)

print("Distribución de entrenamiento")
print("-" * 40)
print(f"sin_ocultamiento: {n_clase_0}")
print(f"ocultamiento:     {n_clase_1}")

print("\nPesos de clase")
print("-" * 40)
print(f"Clase 0 - sin_ocultamiento: {peso_0:.4f}")
print(f"Clase 1 - ocultamiento:     {peso_1:.4f}")

print(
    "\nRelación entre pesos:",
    f"{peso_1 / peso_0:.2f}:1"
)

# ------------------------------------------------------------
# Función de pérdida
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print("\nFunción de pérdida:")
print(criterion)

Distribución de entrenamiento
----------------------------------------
sin_ocultamiento: 570
ocultamiento:     132

Pesos de clase
----------------------------------------
Clase 0 - sin_ocultamiento: 0.6158
Clase 1 - ocultamiento:     2.6591

Relación entre pesos: 4.32:1

Función de pérdida:
CrossEntropyLoss()


### 3.6 Optimizador e hiperparámetros iniciales

La configuración final de fine-tuning contiene dos grupos de parámetros con
diferente grado de conocimiento previo.

El bloque `layer4[1]` conserva pesos aprendidos durante el preentrenamiento en
Kinetics-400. Para evitar modificaciones bruscas de estas representaciones se
utiliza una tasa de aprendizaje relativamente pequeña.

La capa `fc`, en cambio, fue sustituida por un nuevo clasificador con dos
salidas y sus parámetros fueron inicializados aleatoriamente. Por esta razón
se utiliza una tasa de aprendizaje mayor para que el clasificador pueda
adaptarse con mayor rapidez al problema binario.

Se utiliza el optimizador AdamW, que combina actualización adaptativa de los
parámetros con regularización mediante `weight_decay`.

La configuración inicial del baseline es:

| Parámetro                 | Valor    |
| ------------------------- | -------- |
| Optimizador               | AdamW    |
| Learning rate `layer4[1]` | `1×10⁻⁴` |
| Learning rate `fc`        | `1×10⁻³` |
| Weight decay              | `1×10⁻⁴` |
| Batch size                | 16       |
| Precisión mixta           | Sí       |
| Máximo de épocas          | 15       |

Las tasas de aprendizaje se mantienen diferenciadas por un factor de 10 para
permitir una adaptación más rápida del nuevo clasificador sin modificar con
la misma intensidad el bloque preentrenado.

Estos valores constituyen la configuración inicial del baseline. Las decisiones
posteriores sobre checkpoint y detención del entrenamiento se realizarán
exclusivamente utilizando el conjunto de validación; el conjunto de prueba
permanece aislado.

In [34]:
import torch.optim as optim

# ------------------------------------------------------------
# Hiperparámetros iniciales
# ------------------------------------------------------------

LR_LAYER4 = 1e-4
LR_FC = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 15


# ------------------------------------------------------------
# Optimizador con tasas diferenciadas
# ------------------------------------------------------------

optimizer = optim.AdamW(
    [
        {
            "params": model.layer4[1].parameters(),
            "lr": LR_LAYER4
        },
        {
            "params": model.fc.parameters(),
            "lr": LR_FC
        }
    ],
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# Verificación de la configuración
# ------------------------------------------------------------

print("Optimizador:", optimizer.__class__.__name__)
print("Weight decay:", WEIGHT_DECAY)
print("Máximo de épocas:", MAX_EPOCHS)

print("\nGrupos de parámetros")
print("-" * 45)

for i, grupo in enumerate(
    optimizer.param_groups,
    start=1
):
    n_params = sum(
        p.numel()
        for p in grupo["params"]
    )

    print(
        f"Grupo {i}: "
        f"{n_params:,} parámetros | "
        f"lr = {grupo['lr']:.1e}"
    )

Optimizador: AdamW
Weight decay: 0.0001
Máximo de épocas: 15

Grupos de parámetros
---------------------------------------------
Grupo 1: 14,157,824 parámetros | lr = 1.0e-04
Grupo 2: 1,026 parámetros | lr = 1.0e-03


### 3.7 Métricas de evaluación

Las métricas del baseline se definen antes del entrenamiento y mantienen la
estrategia establecida en el Avance 2. Debido al desbalance entre clases, el
desempeño no se evaluará únicamente mediante accuracy.

Las métricas principales son:

- **ROC-AUC:** mide la capacidad del modelo para ordenar ejemplos positivos y
  negativos a través de todos los posibles umbrales de clasificación.
- **PR-AUC:** resume la relación entre precision y recall y recibe especial
  atención debido a que `ocultamiento` es la clase minoritaria.
- **EER (Equal Error Rate):** identifica el punto de operación en el que la
  tasa de falsos positivos y la tasa de falsos negativos son aproximadamente
  iguales. En esta métrica un valor menor indica mejor desempeño.

Como métricas complementarias dependientes de un umbral se calculan:

- accuracy
- precision
- recall o sensibilidad
- specificity
- F1-score
- matriz de confusión

Durante el seguimiento del entrenamiento se utiliza inicialmente un umbral de
0.5 para las métricas dependientes de umbral. ROC-AUC, PR-AUC y EER se calculan
directamente a partir de las probabilidades y no requieren fijar previamente
un umbral operativo.

El conjunto de prueba no se utilizará para seleccionar métricas, checkpoints
ni umbrales. Cualquier decisión posterior sobre el umbral de clasificación se
realizará exclusivamente con el conjunto de validación.

In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve
)

import numpy as np


def calcular_metricas(
    y_true,
    y_prob,
    threshold=0.5
):
    """
    Calcula métricas binarias.

    Parámetros
    ----------
    y_true:
        Etiquetas reales 0/1.

    y_prob:
        Probabilidad estimada de la clase positiva
        `ocultamiento`.

    threshold:
        Umbral para convertir probabilidades en clases.
    """

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    # --------------------------------------------------------
    # Métricas dependientes del umbral
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Métricas independientes del umbral
    # --------------------------------------------------------

    if len(np.unique(y_true)) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_prob
        )

        pr_auc = average_precision_score(
            y_true,
            y_prob
        )

        fpr, tpr, thresholds = roc_curve(
            y_true,
            y_prob
        )

        fnr = 1 - tpr

        idx_eer = np.nanargmin(
            np.abs(fpr - fnr)
        )

        eer = (
            fpr[idx_eer]
            + fnr[idx_eer]
        ) / 2

        threshold_eer = thresholds[
            idx_eer
        ]

    else:
        # Algunas evaluaciones segmentadas por fuente
        # pueden contener una sola clase.
        roc_auc = np.nan
        pr_auc = np.nan
        eer = np.nan
        threshold_eer = np.nan

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "eer": eer,
        "threshold_eer": threshold_eer,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [36]:
y_true_prueba = np.array(
    [0, 0, 0, 1, 1, 1]
)

y_prob_prueba = np.array(
    [0.10, 0.25, 0.40, 0.55, 0.75, 0.90]
)

metricas_prueba = calcular_metricas(
    y_true_prueba,
    y_prob_prueba,
    threshold=0.5
)

for nombre, valor in metricas_prueba.items():
    if isinstance(valor, float):
        print(f"{nombre:15s}: {valor:.4f}")
    else:
        print(f"{nombre:15s}: {valor}")

accuracy       : 1.0000
precision      : 1.0000
recall         : 1.0000
specificity    : 1.0000
f1             : 1.0000
roc_auc        : 1.0000
pr_auc         : 1.0000
eer            : 0.0000
threshold_eer  : 0.5500
tn             : 3
fp             : 0
fn             : 0
tp             : 3


#### 3.7.1 Referencias mínimas de desempeño

No se dispone de un modelo histórico de BNext ni de un nivel mínimo de
desempeño operativo previamente establecido para este problema. Por esta
razón, el baseline se compara inicialmente contra referencias no informativas
derivadas de la distribución de las clases.

Para un clasificador que no contiene información útil sobre la variable
objetivo se consideran las siguientes referencias:

- **ROC-AUC = 0.5:** corresponde al comportamiento esperado de una
  clasificación sin capacidad de discriminación.
- **PR-AUC:** su referencia depende de la prevalencia de la clase positiva.
  Debido a que `ocultamiento` es minoritaria, esta referencia es
  considerablemente menor que 0.5.
- **EER ≈ 0.5:** representa un comportamiento cercano al azar, donde las tasas
  de falsos positivos y falsos negativos son elevadas y similares.
- **Clasificador mayoritario:** siempre predice `sin_ocultamiento`. Puede
  obtener una accuracy elevada debido al desbalance, pero presenta recall y
  F1 iguales a cero para la clase `ocultamiento`.

Para considerar que el baseline ha aprendido una señal discriminativa mínima,
su desempeño sobre validación deberá superar las referencias no informativas
en las métricas independientes de umbral:

$$
\text{ROC-AUC} > 0.5
$$

$$
\text{PR-AUC} > P(y=1)
$$

$$
\text{EER} < 0.5
$$

Estas condiciones representan un criterio de viabilidad experimental y no un
requisito de aceptación para una solución productiva. Un resultado superior a
estas referencias indicaría que el modelo ha aprendido información útil para
distinguir las clases, pero no demostraría por sí mismo que su desempeño sea
suficiente para una implementación operacional.

In [37]:
def resumen_referencias(df_particion, nombre):
    """
    Calcula referencias no informativas a partir de la
    distribución real de una partición.
    """

    n = len(df_particion)

    n_pos = (
        df_particion["clase_binaria"] == 1
    ).sum()

    n_neg = (
        df_particion["clase_binaria"] == 0
    ).sum()

    prevalencia_pos = n_pos / n
    prevalencia_neg = n_neg / n

    # Clasificador mayoritario:
    # siempre predice la clase 0
    accuracy_mayoritario = prevalencia_neg

    return {
        "split": nombre,
        "n": n,
        "positivos": n_pos,
        "negativos": n_neg,
        "prevalencia_ocultamiento": prevalencia_pos,
        "accuracy_mayoritario": accuracy_mayoritario,
        "roc_auc_no_informativo": 0.5,
        "pr_auc_referencia": prevalencia_pos,
        "eer_referencia": 0.5
    }


referencias = pd.DataFrame(
    [
        resumen_referencias(
            df_train,
            "train"
        ),
        resumen_referencias(
            df_val,
            "val"
        ),
        resumen_referencias(
            df_test,
            "test"
        )
    ]
)

display(
    referencias.round(4)
)

,split,n,positivos,negativos,prevalencia_ocultamiento,accuracy_mayoritario,roc_auc_no_informativo,pr_auc_referencia,eer_referencia
0,train,702,132,570,0.1880,0.8120,0.5,0.1880,0.5
1,val,136,30,106,0.2206,0.7794,0.5,0.2206,0.5
2,test,147,28,119,0.1905,0.8095,0.5,0.1905,0.5


### 3.8 Entrenamiento y validación del baseline

El modelo se entrena durante un máximo de 15 épocas utilizando la configuración
definida previamente. En cada época se realiza una fase de entrenamiento y una
fase de validación.

Durante entrenamiento:

- Se actualizan únicamente `layer4[1]` y `fc`
- Los bloques congelados permanecen en modo de evaluación para conservar sus
  estadísticas de Batch Normalization
- Se utiliza `CrossEntropyLoss` ponderada
- Se emplea precisión mixta para reducir el consumo de memoria GPU
- Se registran la pérdida y las métricas de clasificación

Durante validación no se actualizan parámetros. Las probabilidades de
`ocultamiento` se utilizan para calcular ROC-AUC, PR-AUC, EER y las métricas
dependientes del umbral de 0.5.

El mejor checkpoint se selecciona exclusivamente mediante **PR-AUC de
validación**. Esta decisión se establece antes del entrenamiento debido al
desbalance entre clases y a la importancia de evaluar adecuadamente la clase
minoritaria.

Las 15 épocas se conservan en el historial aunque el mejor modelo ocurra antes,
ya que las curvas de entrenamiento y validación se utilizarán posteriormente
para analizar posibles patrones de subajuste o sobreajuste.

El conjunto de prueba permanece completamente aislado durante este proceso.

In [38]:
import time

import torch
import pandas as pd


USE_AMP = device.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


def entrenar_una_epoca(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):
    """
    Ejecuta una época de entrenamiento.
    """

    configurar_modo_entrenamiento(model)

    loss_acumulada = 0.0
    n_ejemplos = 0

    y_true = []
    y_prob = []

    for clips, etiquetas in loader:

        clips = clips.to(
            device,
            non_blocking=True
        )

        etiquetas = etiquetas.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = model(clips)

            loss = criterion(
                logits,
                etiquetas
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        batch_size_actual = (
            etiquetas.size(0)
        )

        loss_acumulada += (
            loss.item()
            * batch_size_actual
        )

        n_ejemplos += batch_size_actual

        probabilidades = (
            torch.softmax(
                logits.detach(),
                dim=1
            )[:, 1]
        )

        y_true.extend(
            etiquetas
            .detach()
            .cpu()
            .numpy()
        )

        y_prob.extend(
            probabilidades
            .cpu()
            .numpy()
        )

    loss_media = (
        loss_acumulada
        / n_ejemplos
    )

    metricas = calcular_metricas(
        y_true,
        y_prob,
        threshold=0.5
    )

    return loss_media, metricas

In [39]:
@torch.no_grad()
def evaluar_una_epoca(
    model,
    loader,
    criterion,
    device
):
    """
    Evalúa el modelo sin actualizar parámetros.
    """

    model.eval()

    loss_acumulada = 0.0
    n_ejemplos = 0

    y_true = []
    y_prob = []

    for clips, etiquetas in loader:

        clips = clips.to(
            device,
            non_blocking=True
        )

        etiquetas = etiquetas.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = model(clips)

            loss = criterion(
                logits,
                etiquetas
            )

        batch_size_actual = (
            etiquetas.size(0)
        )

        loss_acumulada += (
            loss.item()
            * batch_size_actual
        )

        n_ejemplos += batch_size_actual

        probabilidades = (
            torch.softmax(
                logits,
                dim=1
            )[:, 1]
        )

        y_true.extend(
            etiquetas
            .cpu()
            .numpy()
        )

        y_prob.extend(
            probabilidades
            .cpu()
            .numpy()
        )

    loss_media = (
        loss_acumulada
        / n_ejemplos
    )

    metricas = calcular_metricas(
        y_true,
        y_prob,
        threshold=0.5
    )

    return loss_media, metricas

In [40]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    CHECKPOINT_DIR
    / "r3d18_baseline_best.pt"
)

print(
    "Checkpoint:",
    BEST_MODEL_PATH.resolve()
)

Checkpoint: /mnt/c/dev/MNA---Proyecto-Integrador/Notebooks/checkpoints/r3d18_baseline_best.pt


In [41]:
historial = []

mejor_val_pr_auc = -float("inf")
mejor_epoca = None

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    inicio_epoca = time.perf_counter()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # Entrenamiento
    # --------------------------------------------------------

    train_loss, train_metrics = (
        entrenar_una_epoca(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device
        )
    )

    # --------------------------------------------------------
    # Validación
    # --------------------------------------------------------

    val_loss, val_metrics = (
        evaluar_una_epoca(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )
    )

    tiempo_epoca = (
        time.perf_counter()
        - inicio_epoca
    )

    if device.type == "cuda":
        memoria_pico = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )
    else:
        memoria_pico = float("nan")

    # --------------------------------------------------------
    # Historial
    # --------------------------------------------------------

    registro = {
        "epoch": epoch,

        "train_loss": train_loss,
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "train_roc_auc": train_metrics["roc_auc"],
        "train_pr_auc": train_metrics["pr_auc"],
        "train_eer": train_metrics["eer"],

        "val_loss": val_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_eer": val_metrics["eer"],

        "tiempo_s": tiempo_epoca,
        "gpu_peak_gb": memoria_pico
    }

    historial.append(registro)

    # --------------------------------------------------------
    # Selección del mejor checkpoint
    # --------------------------------------------------------

    if (
        val_metrics["pr_auc"]
        > mejor_val_pr_auc
    ):
        mejor_val_pr_auc = (
            val_metrics["pr_auc"]
        )

        mejor_epoca = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "val_pr_auc":
                    mejor_val_pr_auc,
                "seed":
                    SEED_MODEL,
                "batch_size":
                    BATCH_SIZE,
                "lr_layer4":
                    LR_LAYER4,
                "lr_fc":
                    LR_FC,
                "weight_decay":
                    WEIGHT_DECAY
            },
            BEST_MODEL_PATH
        )

    # --------------------------------------------------------
    # Salida de seguimiento
    # --------------------------------------------------------

    print(
        f"Época {epoch:02d}/{MAX_EPOCHS} | "
        f"{tiempo_epoca:6.1f} s | "
        f"Train loss {train_loss:.4f} | "
        f"Val loss {val_loss:.4f} | "
        f"Val PR-AUC {val_metrics['pr_auc']:.4f} | "
        f"Val ROC-AUC {val_metrics['roc_auc']:.4f} | "
        f"Val F1 {val_metrics['f1']:.4f} | "
        f"Val recall {val_metrics['recall']:.4f}"
    )


historial_df = pd.DataFrame(
    historial
)

print("\nEntrenamiento terminado.")
print(
    "Mejor época:",
    mejor_epoca
)

print(
    "Mejor PR-AUC de validación:",
    f"{mejor_val_pr_auc:.4f}"
)

Época 01/15 |  256.3 s | Train loss 0.5349 | Val loss 0.6084 | Val PR-AUC 0.5911 | Val ROC-AUC 0.7179 | Val F1 0.4638 | Val recall 0.5333
Época 02/15 |  271.4 s | Train loss 0.2454 | Val loss 0.7802 | Val PR-AUC 0.5966 | Val ROC-AUC 0.7390 | Val F1 0.5185 | Val recall 0.4667
Época 03/15 |  253.5 s | Train loss 0.1795 | Val loss 1.0124 | Val PR-AUC 0.5956 | Val ROC-AUC 0.7478 | Val F1 0.5306 | Val recall 0.4333
Época 04/15 |  265.8 s | Train loss 0.1258 | Val loss 1.1719 | Val PR-AUC 0.6052 | Val ROC-AUC 0.7585 | Val F1 0.5283 | Val recall 0.4667
Época 05/15 |  275.5 s | Train loss 0.0780 | Val loss 1.1148 | Val PR-AUC 0.6345 | Val ROC-AUC 0.8013 | Val F1 0.5714 | Val recall 0.4667
Época 06/15 |  266.6 s | Train loss 0.0454 | Val loss 1.4250 | Val PR-AUC 0.5890 | Val ROC-AUC 0.7428 | Val F1 0.5600 | Val recall 0.4667
Época 07/15 |  277.1 s | Train loss 0.0403 | Val loss 1.5228 | Val PR-AUC 0.6068 | Val ROC-AUC 0.7664 | Val F1 0.5217 | Val recall 0.4000
Época 08/15 |  267.7 s | Train los

In [42]:
print("Mejor época:", mejor_epoca)
print("Mejor Val PR-AUC:", mejor_val_pr_auc)
print("Checkpoint:", BEST_MODEL_PATH.resolve())
print("Existe:", BEST_MODEL_PATH.exists())

if BEST_MODEL_PATH.exists():
    print(
        "Tamaño:",
        f"{BEST_MODEL_PATH.stat().st_size / 1024**2:.1f} MB"
    )

Mejor época: 5
Mejor Val PR-AUC: 0.6344950319579342
Checkpoint: /mnt/c/dev/MNA---Proyecto-Integrador/Notebooks/checkpoints/r3d18_baseline_best.pt
Existe: True
Tamaño: 234.6 MB


In [43]:
HISTORIAL_PATH = CHECKPOINT_DIR / "r3d18_baseline_historial.csv"

historial_df.to_csv(
    HISTORIAL_PATH,
    index=False
)

print("Historial guardado en:")
print(HISTORIAL_PATH.resolve())
print("Existe:", HISTORIAL_PATH.exists())

Historial guardado en:
/mnt/c/dev/MNA---Proyecto-Integrador/Notebooks/checkpoints/r3d18_baseline_historial.csv
Existe: True
